# 02 · QLoRA Fine-Tuning with Unsloth on a free T4

Fine-tunes **Qwen3-4B-Instruct-2507** (Apache-2.0) with **4-bit QLoRA** to emit the strict JSON contract built in notebook 01, then exports a **GGUF `q4_k_m`** model for Ollama / llama.cpp.

| | |
|---|---|
| Hardware | Free Colab **T4 (15 GB)** — `Runtime → Change runtime type → T4 GPU` |
| Precision | 4-bit NF4 base weights, LoRA in fp16 (`fp16=True`, `bf16=False` — T4 has no bf16) |
| Memory tricks | Unsloth gradient checkpointing, 8-bit AdamW, batch 2 × grad-accum 4 |
| Loss | **Assistant tokens only** (`train_on_responses_only`) — the long system prompt isn't learned, only the JSON answer |
| Resilience | Checkpoints go to Google Drive; re-running the notebook **resumes** after a Colab disconnect |
| Outputs | LoRA adapter repo + GGUF repo on Hugging Face Hub, optional W&B run |

**Before you start:** Colab 🔑 Secrets → `HF_TOKEN` (write scope). Optional: `WANDB_API_KEY` (free account at wandb.ai).

## 0. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


## 1. Install
Unsloth pins compatible `torch` / `transformers` / `trl` / `peft` / `bitsandbytes` versions itself, so we let it resolve them and **record the exact versions** in the model card for reproducibility (pinning by hand breaks every time Colab updates its base image).

In [ ]:
%pip install -q unsloth
%pip install -q "wandb>=0.17"

In [ ]:
import unsloth  # must be imported before transformers / trl / peft so its patches apply
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

import gc
import inspect
import json
import os
import re
import time
from pathlib import Path
from typing import List, Literal, Optional

import torch
import transformers
import trl
import peft
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download, login, whoami

VERSIONS = {
    "unsloth": unsloth.__version__, "torch": torch.__version__, "transformers": transformers.__version__,
    "trl": trl.__version__, "peft": peft.__version__, "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
}
print(json.dumps(VERSIONS, indent=2))
assert torch.cuda.is_available(), "No GPU — switch the runtime to T4 GPU."

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1115 15:29:43.183000 464 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 15:29:43.307000 464 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
{
  "unsloth": "2026.11.4",
  "torch": "2.14.1+cu130",
  "transformers": "5.17.0",
  "trl": "1.15.0",
  "peft": "0.21.2",
  "cuda": "13.0",
  "gpu": "Tesla T4"
}


## 2. Configuration

In [ ]:
SEED = 3407

# ---- Model ----
BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"   # Apache-2.0, non-thinking instruct model
MAX_SEQ_LENGTH = 1024        # notebook 01 measured max 567 tokens; 1024 leaves headroom and saves VRAM

# ---- LoRA ----
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0           # 0 enables Unsloth's fast path
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ---- Training (tuned for a 15 GB T4) ----
NUM_EPOCHS = 2
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 4               # effective batch = 8
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
EVAL_SUBSET = 100            # validation rows used for in-training eval (keeps eval fast)
EVAL_STEPS = 50
SAVE_STEPS = 100             # must be a multiple of EVAL_STEPS for load_best_model_at_end

# ---- Experiment tracking: "online" | "offline" | "disabled" ----
WANDB_MODE = "online"
WANDB_PROJECT = "slm-pii-masker"
RUN_NAME = f"qwen3-4b-qlora-r{LORA_R}-e{NUM_EPOCHS}-{time.strftime('%Y%m%d-%H%M')}"

# ---- Persistence ----
USE_DRIVE = True             # checkpoints survive Colab disconnects
PUSH_TO_HUB = True
EXPORT_GGUF = True
GGUF_QUANT = "q4_k_m"

# ---- Names ----
DATASET_NAME = "enterprise-pii-ticket-sft"
MODEL_SLUG = "pii-masker-qwen3-4b"

In [ ]:
def get_secret(name: str) -> Optional[str]:
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name)


HF_TOKEN = get_secret("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN missing: Colab 🔑 Secrets → add HF_TOKEN (write scope) and enable notebook access.")
login(token=HF_TOKEN, add_to_git_credential=False)
HF_USERNAME = whoami(token=HF_TOKEN)["name"]

HF_DATASET_REPO = f"{HF_USERNAME}/{DATASET_NAME}"
LORA_REPO = f"{HF_USERNAME}/{MODEL_SLUG}-lora"
GGUF_REPO = f"{HF_USERNAME}/{MODEL_SLUG}-GGUF"
print(f"Dataset : {HF_DATASET_REPO}\nLoRA    : {LORA_REPO}\nGGUF    : {GGUF_REPO}")

# ---- Weights & Biases ----
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
os.environ["WANDB_LOG_MODEL"] = "false"
if WANDB_MODE == "online":
    wandb_key = get_secret("WANDB_API_KEY")
    if wandb_key:
        import wandb
        wandb.login(key=wandb_key, relogin=True)
    else:
        print("⚠️  WANDB_API_KEY not found → logging W&B offline (sync later with `wandb sync`).")
        WANDB_MODE = "offline"
os.environ["WANDB_MODE"] = WANDB_MODE
REPORT_TO = "none" if WANDB_MODE == "disabled" else "wandb"
print(f"W&B mode: {WANDB_MODE}")

# ---- Checkpoint directory ----
if USE_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        OUTPUT_DIR = Path("/content/drive/MyDrive/slm-pii-masker/checkpoints") / MODEL_SLUG
    except Exception as e:
        print(f"⚠️  Drive unavailable ({e}); checkpoints stay on the ephemeral VM.")
        OUTPUT_DIR = Path("outputs") / MODEL_SLUG
else:
    OUTPUT_DIR = Path("outputs") / MODEL_SLUG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoints: {OUTPUT_DIR}")

Dataset : Zain9989/enterprise-pii-ticket-sft
LoRA    : Zain9989/pii-masker-qwen3-4b-lora
GGUF    : Zain9989/pii-masker-qwen3-4b-GGUF


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B mode: online
Mounted at /content/drive
Checkpoints: /content/drive/MyDrive/slm-pii-masker/checkpoints/pii-masker-qwen3-4b


## 3. Load the dataset and the shared output contract
`prompt_config.json` was published by notebook 01 — we validate against **exactly** the same labels.

In [ ]:
ds = load_dataset(HF_DATASET_REPO, token=HF_TOKEN)
print(ds)

PROMPT_CONFIG = json.loads(Path(hf_hub_download(
    HF_DATASET_REPO, "prompt_config.json", repo_type="dataset", token=HF_TOKEN)).read_text())
SYSTEM_PROMPT = PROMPT_CONFIG["system_prompt"]
USER_TEMPLATE = PROMPT_CONFIG["user_template"]
PII_TAGS = PROMPT_CONFIG["pii_tags"]

from pydantic import BaseModel, ConfigDict, ValidationError, field_validator, model_validator

TAG_PATTERN = re.compile(r"\[(" + "|".join(PII_TAGS) + r")\]")


class TicketAnalysis(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    masked_text: str
    detected_pii_types: List[Literal["PERSON", "EMAIL", "PHONE", "SSN", "CREDIT_CARD", "IP_ADDRESS", "API_KEY"]]
    ticket_category: Literal["Billing_and_Payments", "Technical_Support", "Security_and_Access",
                             "Legal_and_Compliance", "General_Inquiry"]
    urgency_level: Literal["Low", "Medium", "High"]

    @field_validator("detected_pii_types")
    @classmethod
    def _unique(cls, v):
        if len(set(v)) != len(v):
            raise ValueError("duplicate PII types")
        return v

    @model_validator(mode="after")
    def _consistent(self):
        if set(TAG_PATTERN.findall(self.masked_text)) != set(self.detected_pii_types):
            raise ValueError("tags in masked_text do not match detected_pii_types")
        return self


# Guard against the schema here drifting from notebook 01's contract.
_schema_fields = TicketAnalysis.model_fields
assert list(_schema_fields["detected_pii_types"].annotation.__args__[0].__args__) == PII_TAGS
assert list(_schema_fields["ticket_category"].annotation.__args__) == PROMPT_CONFIG["categories"]
assert list(_schema_fields["urgency_level"].annotation.__args__) == PROMPT_CONFIG["urgency_levels"]
print("✅ Local schema matches prompt_config.json")

README.md:   0%|          | 0.00/3.05k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.23MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  559kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  565kB            

... (3 outputs left out)

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages', 'source_text', 'target_json', 'masked_text', 'detected_pii_types', 'ticket_category', 'urgency_level', 'origin', 'n_tokens'],
        num_rows: 3400
    })
    validation: Dataset({
        features: ['messages', 'source_text', 'target_json', 'masked_text', 'detected_pii_types', 'ticket_category', 'urgency_level', 'origin', 'n_tokens'],
        num_rows: 300
    })
    test: Dataset({
        features: ['messages', 'source_text', 'target_json', 'masked_text', 'detected_pii_types', 'ticket_category', 'urgency_level', 'origin', 'n_tokens'],
        num_rows: 300
    })
})


prompt_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

✅ Local schema matches prompt_config.json


## 4. Load the 4-bit base model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,              # auto → float16 on T4
    load_in_4bit=True,
    token=HF_TOKEN,
)

# Plain ChatML, no <think> blocks and no injected default system prompt.
# This exact template is used for training, inference, and gets embedded in the GGUF for Ollama/llama.cpp.
CHATML_TEMPLATE = (
    "{%- for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\\n' + message['content'] + '<|im_end|>' + '\\n' }}"
    "{%- endfor %}"
    "{%- if add_generation_prompt %}{{ '<|im_start|>assistant\\n' }}{%- endif %}"
)
_original = tokenizer.apply_chat_template(ds["train"][0]["messages"], tokenize=False)
print("Original template inserted <think>:", "<think>" in _original)
tokenizer.chat_template = CHATML_TEMPLATE
tokenizer.eos_token = "<|im_end|>"
if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    tokenizer.pad_token = "<|endoftext|>"

assert not is_bfloat16_supported(), "This notebook is tuned for a T4 (fp16). On Ampere+ set fp16=False, bf16=True."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GB | "
      f"reserved after load: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")

_preview = tokenizer.apply_chat_template(ds["train"][0]["messages"], tokenize=False, add_generation_prompt=False)
print(_preview[-400:])
assert "<|im_start|>assistant\n{" in _preview, "assistant turn should start directly with the JSON"
assert "<think>" not in _preview
print("✅ Chat template OK")

==((====))==  Unsloth 2026.11.4: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Original template inserted <think>: True
GPU: Tesla T4 | VRAM: 14.6 GB | reserved after load: 6.62 GB
ars experiences. Please remember to bring 0817172998910328.
</ticket><|im_end|>
<|im_start|>assistant
{"masked_text": "[PERSON], confirming your therapy for next Monday at 17. Looking forward to learning about your 89 years experiences. Please remember to bring [CREDIT_CARD].", "detected_pii_types": ["PERSON", "CREDIT_CARD"], "ticket_category": "General_Inquiry", "urgency_level": "Low"}<|im_end|>

✅ Chat template OK


## 5. Attach LoRA adapters
Rank-16 adapters on every attention + MLP projection (~33M trainable params, <1% of the model). Small rank, few epochs and a low LR-to-rank ratio limit **catastrophic forgetting** — the base weights stay frozen in 4-bit.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    use_gradient_checkpointing="unsloth",   # offloads activations → large VRAM savings on T4
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

Unsloth 2026.11.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


## 6. Format: ChatML text

In [ ]:
def to_text(batch):
    return {"text": [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in batch["messages"]
    ]}


train_ds = ds["train"].map(to_text, batched=True, remove_columns=ds["train"].column_names)
eval_ds = (ds["validation"].shuffle(seed=SEED).select(range(min(EVAL_SUBSET, len(ds["validation"]))))
           .map(to_text, batched=True, remove_columns=ds["validation"].column_names))


def within_length(batch):
    return [len(ids) <= MAX_SEQ_LENGTH for ids in tokenizer(batch["text"], add_special_tokens=False)["input_ids"]]


n_before = len(train_ds)
train_ds = train_ds.filter(within_length, batched=True)
eval_ds = eval_ds.filter(within_length, batched=True)
print(f"train={len(train_ds):,} (dropped {n_before - len(train_ds)} over {MAX_SEQ_LENGTH} tokens) | eval={len(eval_ds)}")

Map:   0%|          | 0/3400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3400 [00:00<?, ? examples/s]

Filter:   0%|          | 0/100 [00:00<?, ? examples/s]

train=3,400 (dropped 0 over 1024 tokens) | eval=100


## 7. Trainer
Built so it works across TRL versions (`max_seq_length` → `max_length`, `tokenizer` → `processing_class`).

In [ ]:
from trl import SFTConfig, SFTTrainer

_cfg_params = inspect.signature(SFTConfig.__init__).parameters
cfg = dict(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_NAME,
    dataset_text_field="text",
    packing=False,
    dataset_num_proc=1,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    weight_decay=WEIGHT_DECAY,
    optim="adamw_8bit",
    fp16=True,
    bf16=False,
    max_grad_norm=1.0,
    logging_steps=10,
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to=REPORT_TO,
    seed=SEED,
)
cfg["max_length" if "max_length" in _cfg_params else "max_seq_length"] = MAX_SEQ_LENGTH
cfg["eval_strategy" if "eval_strategy" in _cfg_params else "evaluation_strategy"] = "steps"
sft_config = SFTConfig(**{k: v for k, v in cfg.items() if k in _cfg_params})

_trainer_params = inspect.signature(SFTTrainer.__init__).parameters
tok_kw = "processing_class" if "processing_class" in _trainer_params else "tokenizer"
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    **{tok_kw: tokenizer},
)

# Only the assistant's JSON contributes to the loss.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Verify the masking: decoded labels must contain only the JSON answer.
if "labels" in trainer.train_dataset.column_names:
    _labels = trainer.train_dataset[0]["labels"]
    _visible = tokenizer.decode([t for t in _labels if t != -100])
    print("Loss is computed on:\n", _visible[:400])
    assert _visible.lstrip().startswith("{") and "You are an enterprise" not in _visible, \
        "Response-only masking failed — the prompt is leaking into the loss."
else:
    print("ℹ️  This Unsloth version masks labels inside the data collator; skipping the label preview.")

steps_per_epoch = -(-len(train_ds) // (PER_DEVICE_BATCH * GRAD_ACCUM))
print(f"≈{steps_per_epoch} optimizer steps/epoch → {steps_per_epoch * NUM_EPOCHS} total")

Unsloth: Tokenizing ["text"]:   0%|          | 0/3400 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/100 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/3400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Loss is computed on:
 {"masked_text": "[PERSON], confirming your therapy for next Monday at 17. Looking forward to learning about your 89 years experiences. Please remember to bring [CREDIT_CARD].", "detected_pii_types": ["PERSON", "CREDIT_CARD"], "ticket_category": "General_Inquiry", "urgency_level": "Low"}<|im_end|>
≈425 optimizer steps/epoch → 850 total


## 8. Train (auto-resumes from the latest checkpoint)
Expect roughly **1 hour** for 2 epochs on a T4. If Colab disconnects, reconnect and **Run all** again — it continues from the last checkpoint on Drive.

In [ ]:
def latest_checkpoint(path: Path) -> Optional[str]:
    ckpts = sorted(path.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
    return str(ckpts[-1]) if ckpts else None


resume_from = latest_checkpoint(OUTPUT_DIR)
print(f"Resuming from {resume_from}" if resume_from else "Starting fresh")

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=resume_from)
train_minutes = (time.time() - t0) / 60
peak_vram_gb = torch.cuda.max_memory_reserved() / 1024**3

eval_metrics = trainer.evaluate()
history = trainer.state.log_history
TRAINING_SUMMARY = {
    "base_model": BASE_MODEL,
    "dataset": HF_DATASET_REPO,
    "train_rows": len(train_ds),
    "epochs": NUM_EPOCHS,
    "global_steps": trainer.state.global_step,
    "effective_batch_size": PER_DEVICE_BATCH * GRAD_ACCUM,
    "learning_rate": LEARNING_RATE,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "targets": LORA_TARGETS},
    "final_train_loss": round(float(train_result.training_loss), 4),
    "best_eval_loss": round(float(trainer.state.best_metric), 4) if trainer.state.best_metric is not None else None,
    "final_eval_loss": round(float(eval_metrics["eval_loss"]), 4),
    "train_minutes_this_session": round(train_minutes, 1),
    "peak_vram_gb": round(peak_vram_gb, 2),
    "gpu": VERSIONS["gpu"],
    "versions": VERSIONS,
    "wandb_run": RUN_NAME if REPORT_TO == "wandb" else None,
}
print(json.dumps(TRAINING_SUMMARY, indent=2))
(OUTPUT_DIR / "training_summary.json").write_text(json.dumps(TRAINING_SUMMARY, indent=2))

Starting fresh


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,400 | Num Epochs = 2 | Total steps = 850
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)
wandb: Currently logged in as: za488778 (za488778-zelulabs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.026057,0.023714
100,0.015393,0.018397
150,0.010009,0.015412
200,0.014117,0.011737
250,0.010239,0.011873
300,0.009317,0.009525
350,0.008235,0.008281
400,0.005074,0.009307
450,0.006075,0.007060
500,0.001920,0.006014


Filter:   0%|          | 0/100 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen3ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/slm-pii-masker/checkpoints/pii-masker-qwen3-4b/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/slm-pii-masker/checkpoints/pii-masker-qwen3-4b/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/slm-pii-masker/checkpoints/pii-masker-qwen3-4b/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/slm-pii-masker/checkpoints/pii-masker-qwen3-4b/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/slm-pii-

Step,Training Loss,Validation Loss
50,0.026057,0.023714
100,0.015393,0.018397
150,0.010009,0.015412
200,0.014117,0.011737
250,0.010239,0.011873
300,0.009317,0.009525
350,0.008235,0.008281
400,0.005074,0.009307
450,0.006075,0.007060
500,0.001920,0.006014


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/slm-pii-masker/checkpoints/pii-masker-qwen3-4b/checkpoint-700/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/slm-pii-masker/checkpoints/pii-masker-qwen3-4b/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/slm-pii-masker/checkpoints/pii-masker-qwen3-4b/checkpoint-850/tokenizer_config.json.


Training Loss,Validation Loss,Step
0.002071,0.003581,850


{
  "base_model": "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",
  "dataset": "Zain9989/enterprise-pii-ticket-sft",
  "train_rows": 3400,
  "epochs": 2,
  "global_steps": 850,
  "effective_batch_size": 8,
  "learning_rate": 0.0002,
  "lora": {
    "r": 16,
    "alpha": 32,
    "dropout": 0.0,
    "targets": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ]
  },
  "final_train_loss": 0.0145,
  "best_eval_loss": 0.0036,
  "final_eval_loss": 0.0036,
  "train_minutes_this_session": 95.2,
  "peak_vram_gb": 4.05,
  "gpu": "Tesla T4",
  "versions": {
    "unsloth": "2026.11.4",
    "torch": "2.14.1+cu130",
    "transformers": "5.17.0",
    "trl": "1.15.0",
    "peft": "0.21.2",
    "cuda": "13.0",
    "gpu": "Tesla T4"
  },
  "wandb_run": "qwen3-4b-qlora-r16-e2-20261115-1530"
}


861

## 9. Smoke test on held-out tickets
A quick check that the model emits schema-valid JSON. The rigorous base-vs-fine-tuned benchmark is notebook 03.

In [ ]:
FastLanguageModel.for_inference(model)


def extract_json(text: str) -> Optional[dict]:
    """Strip code fences / chatter and parse the first top-level JSON object."""
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    start = text.find("{")
    if start == -1:
        return None
    depth, in_str, esc = 0, False, False
    for i, ch in enumerate(text[start:], start):
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except json.JSONDecodeError:
                    return None
    return None


@torch.inference_mode()
def analyze(ticket: str) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_TEMPLATE.format(ticket=ticket)}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = inputs["input_ids"].shape[1]
    out = model.generate(
        **inputs,
        max_new_tokens=min(1024, n_in + 128),
        do_sample=False, temperature=None, top_p=None, top_k=None,
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(out[0, n_in:], skip_special_tokens=True)


test_rows = ds["test"].shuffle(seed=SEED).select(range(20))
valid, cat_ok = 0, 0
for i, row in enumerate(test_rows):
    raw = analyze(row["source_text"])
    parsed = extract_json(raw)
    try:
        pred = TicketAnalysis.model_validate(parsed) if parsed is not None else None
    except ValidationError:
        pred = None
    valid += pred is not None
    cat_ok += pred is not None and pred.ticket_category == row["ticket_category"]
    if i < 2:
        print("INPUT :", row["source_text"][:300])
        print("OUTPUT:", raw[:500], "\n")
print(f"Smoke test: {valid}/20 schema-valid JSON | {cat_ok}/20 correct department")
TRAINING_SUMMARY["smoke_test"] = {"schema_valid": valid, "category_correct": cat_ok, "n": 20}

Both `max_new_tokens` (=412) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=418) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INPUT : Hi Ardith, for our records and court documentation, could you verify your current residential address is Fayette County, Apt. 924?
OUTPUT: {"masked_text": "Hi [PERSON], for our records and court documentation, could you verify your current residential address is Fayette County, Apt. 924?", "detected_pii_types": ["PERSON"], "ticket_category": "Legal_and_Compliance", "urgency_level": "Low"} 



Both `max_new_tokens` (=424) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INPUT : Reba, as a follow-up to our previous communication, I have forwarded an overview of stress management techniques and resources to your email: Hilton_Bins78@gmail.com.
OUTPUT: {"masked_text": "[PERSON], as a follow-up to our previous communication, I have forwarded an overview of stress management techniques and resources to your email: [EMAIL].", "detected_pii_types": ["PERSON", "EMAIL"], "ticket_category": "General_Inquiry", "urgency_level": "Medium"} 



Both `max_new_tokens` (=425) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=460) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=425) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=417) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Smoke test: 20/20 schema-valid JSON | 20/20 correct department


## 10. Push the LoRA adapter to the Hub

In [ ]:
if PUSH_TO_HUB:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(LORA_REPO, repo_type="model", exist_ok=True, private=False)
    model.push_to_hub(LORA_REPO, token=HF_TOKEN, commit_message="Upload QLoRA adapter")
    tokenizer.push_to_hub(LORA_REPO, token=HF_TOKEN, commit_message="Upload tokenizer")

    lora_card = f"""---
license: apache-2.0
base_model: {BASE_MODEL}
datasets:
- {HF_DATASET_REPO}
library_name: peft
pipeline_tag: text-generation
tags: [unsloth, qlora, pii, redaction, ticket-classification, json, trl]
---

# {MODEL_SLUG} — QLoRA adapter

LoRA adapter that turns **Qwen3-4B-Instruct-2507** into an enterprise **PII masker + ticket router** returning strict JSON
(`masked_text`, `detected_pii_types`, `ticket_category`, `urgency_level`).

* Trained on a free Colab **T4** with Unsloth + TRL `SFTTrainer`, 4-bit QLoRA, fp16, loss on assistant tokens only.
* Dataset: [`{HF_DATASET_REPO}`](https://huggingface.co/datasets/{HF_DATASET_REPO})
* Quantized GGUF for Ollama / llama.cpp: [`{GGUF_REPO}`](https://huggingface.co/{GGUF_REPO})

## Usage
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained("{LORA_REPO}", max_seq_length={MAX_SEQ_LENGTH}, load_in_4bit=True)
FastLanguageModel.for_inference(model)
```
The system prompt is in `prompt_config.json` in this repo — use it verbatim.

## Training summary
```json
{json.dumps(TRAINING_SUMMARY, indent=2)}
```
"""
    api.upload_file(path_or_fileobj=lora_card.encode(), path_in_repo="README.md", repo_id=LORA_REPO,
                    commit_message="Model card")
    api.upload_file(path_or_fileobj=json.dumps(PROMPT_CONFIG, indent=2).encode(), path_in_repo="prompt_config.json",
                    repo_id=LORA_REPO, commit_message="Add prompt config")
    api.upload_file(path_or_fileobj=json.dumps(TRAINING_SUMMARY, indent=2).encode(),
                    path_in_repo="training_summary.json", repo_id=LORA_REPO, commit_message="Add training summary")
    print(f"✅ Adapter live at https://huggingface.co/{LORA_REPO}")

README.md:   0%|          | 0.00/2.16k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  12%|#2        | 16.0MB /  132MB            

Validating                    :           |   0%            

Saved model to https://huggingface.co/Zain9989/pii-masker-qwen3-4b-lora


... (4 outputs left out)

  ...mpgrlfyg5c/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


TypeError: HfApi.upload_file() got an unexpected keyword argument 'commit_message'. Did you mean 'commit_message'?

## 11. Export GGUF `q4_k_m` and push
Unsloth merges the adapter into 16-bit weights, builds llama.cpp and quantizes. Takes ~10–15 min and needs ~10 GB of disk. We free the trainer first so the merge has room.

In [ ]:
if EXPORT_GGUF:
    del trainer
    gc.collect()
    torch.cuda.empty_cache()

    GGUF_DIR = Path("gguf_export")
    export_start = time.time()
    model.save_pretrained_gguf(str(GGUF_DIR), tokenizer, quantization_method=GGUF_QUANT)

    candidates = [p for p in Path(".").rglob("*.gguf")
                  if p.stat().st_mtime >= export_start - 5 and GGUF_QUANT in p.name.lower()]
    if not candidates:
        raise FileNotFoundError(f"No *{GGUF_QUANT}*.gguf produced; files found: {list(Path('.').rglob('*.gguf'))}")
    gguf_path = max(candidates, key=lambda p: p.stat().st_size)
    gguf_name = f"{MODEL_SLUG}-{GGUF_QUANT.upper()}.gguf"
    print(f"GGUF: {gguf_path} ({gguf_path.stat().st_size / 1024**3:.2f} GB) → {gguf_name}")

    if PUSH_TO_HUB:
        api = HfApi(token=HF_TOKEN)
        api.create_repo(GGUF_REPO, repo_type="model", exist_ok=True, private=False)
        api.upload_file(path_or_fileobj=str(gguf_path), path_in_repo=gguf_name, repo_id=GGUF_REPO,
                        commit_message=f"Upload {GGUF_QUANT} GGUF")
        gguf_card = f"""---
license: apache-2.0
base_model: {LORA_REPO}
tags: [gguf, llama.cpp, ollama, pii, redaction, unsloth]
---

# {MODEL_SLUG} — GGUF ({GGUF_QUANT.upper()})

Quantized, merged version of [`{LORA_REPO}`](https://huggingface.co/{LORA_REPO}) for CPU/GPU inference with Ollama or llama.cpp.

## Run with Ollama
```bash
ollama run hf.co/{GGUF_REPO}:{GGUF_QUANT.upper()}
```

## Run with llama.cpp
```bash
llama-server -hf {GGUF_REPO}:{GGUF_QUANT.upper()} --ctx-size {MAX_SEQ_LENGTH * 2} --temp 0
```

Always send the system prompt from `prompt_config.json` and use temperature 0.
"""
        api.upload_file(path_or_fileobj=gguf_card.encode(), path_in_repo="README.md", repo_id=GGUF_REPO,
                        commit_message="Model card")
        api.upload_file(path_or_fileobj=json.dumps(PROMPT_CONFIG, indent=2).encode(),
                        path_in_repo="prompt_config.json", repo_id=GGUF_REPO, commit_message="Add prompt config")
        print(f"✅ GGUF live at https://huggingface.co/{GGUF_REPO}")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in gguf_export/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (13 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  ...Instruct-2507.Q4_K_M.gguf:   1%|          | 15.9MB / 2.50GB            

Validating                    :           |   0%            

✅ GGUF live at https://huggingface.co/Zain9989/pii-masker-qwen3-4b-GGUF


## 12. Wrap up

In [ ]:
if REPORT_TO == "wandb":
    import wandb
    if wandb.run is not None:
        wandb.run.summary.update({k: v for k, v in TRAINING_SUMMARY.items() if isinstance(v, (int, float, str))})
        wandb.finish()

print(json.dumps({k: v for k, v in TRAINING_SUMMARY.items() if k != "versions"}, indent=2))
print("\nNext: notebook 03 benchmarks the base model vs this adapter on 200 held-out tickets.")

eval/loss,█▆▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
eval/runtime,▇▄▅█▄▅▁▃▂▄▄▅█▅▆▇▇▇
eval/samples_per_second,▂▅▄▁▅▄█▆▇▅▅▄▁▄▃▂▂▂
eval/steps_per_second,▂▅▄▁▅▅█▆▇▅▅▅▁▄▃▂▂▂
train/epoch,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▁▁▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇██
train/grad_norm,█▃▂▄▂▂▃▂▂▂▂▂▂▁▂▃▂▁▂▁▂▃▂▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▃▃▅███████▇▇▇▇▇▆▆▆▆▆▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
train/loss,█▂▂▂▁▂▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
base_model,unsloth/Qwen3-4B-Ins...
best_eval_loss,0.0036


{
  "base_model": "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",
  "dataset": "Zain9989/enterprise-pii-ticket-sft",
  "train_rows": 3400,
  "epochs": 2,
  "global_steps": 850,
  "effective_batch_size": 8,
  "learning_rate": 0.0002,
  "lora": {
    "r": 16,
    "alpha": 32,
    "dropout": 0.0,
    "targets": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ]
  },
  "final_train_loss": 0.0145,
  "best_eval_loss": 0.0036,
  "final_eval_loss": 0.0036,
  "train_minutes_this_session": 95.2,
  "peak_vram_gb": 4.05,
  "gpu": "Tesla T4",
  "wandb_run": "qwen3-4b-qlora-r16-e2-20261115-1530",
  "smoke_test": {
    "schema_valid": 20,
    "category_correct": 20,
    "n": 20
  }
}

Next: notebook 03 benchmarks the base model vs this adapter on 200 held-out tickets.
